In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/My Drive

/content/drive/My Drive


In [ ]:
# Import libraries
import os
import numpy as np
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

# Set OpenAI API key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
HF_API_KEY = userdata.get('HUGGINGFACE_API_KEY')
os.environ["HUGGINGFACE_API_KEY"] = HF_API_KEY
serper_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key
serp_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serp_api_key
print("✓ Setup complete!")

✓ Setup complete!


In [ ]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
QAmodels = {
    "gpt-4.1-nano": "gpt-4.1-nano",
    "gpt-4o-mini": "gpt-4o-mini"
}

Load Required Libraries

In [ ]:
!pip show langchain

Name: langchain
Version: 1.2.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 85.5 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain langchain-openai
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters

In [ ]:
!pip show transformers

Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: peft, sentence-transformers


Evaluation

Install Required Libraries

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install rouge

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1de574fe75009020f3590f0d8a2d8d0222b3341b3c12cdfe969f5c04a8272873
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
!pip install bleu

  Preparing metadata (setup.py) ... done
  Created wheel for bleu: filename=bleu-0.3-py3-none-any.whl size=5780 sha256=76cdbace604972e7a58cd5d55966a0d92cc5841770d072d237b59c342095e1fd
  Stored in directory: /root/.cache/pip/wheels/a0/08/b2/15cf219d07cdbbc5b1be2d863de6c87e04f99274098e22d061
Successfully built bleu


In [ ]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")

In [ ]:
!python -c "import evaluate; print(evaluate.load('exact_match').compute(references=['hello'], predictions=['hello']))"

{'exact_match': np.float64(1.0)}


In [ ]:
accuracy = evaluate.load("accuracy")
accuracy.description

'\nAccuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:\nAccuracy = (TP + TN) / (TP + TN + FP + FN)\n Where:\nTP: True positive\nTN: True negative\nFP: False positive\nFN: False negative\n'

Metrics for Question Answering

Exact Match (EM):

In [ ]:
import evaluate

# Load the exact_match metric
# Eventually, these will be replaced by actual fact checking question answers

exact_match_metric = evaluate.load("exact_match")

# Sample predictions and references for question answering
# In QA, references are the ground truth answers, and predictions are the model's answers.
predictions = ["The capital of France is Paris.", "The Earth is round.", "Albert Einstein is known for the theory of relativity."]
references = ["The capital of France is Paris.", "The Earth is flat.", "Albert Einstein is known for the theory of relativity."]

# Compute the exact match score
results = exact_match_metric.compute(predictions=predictions, references=references)

print("Exact Match (EM) results:")
print(results)

Exact Match (EM) results:
{'exact_match': np.float64(0.6666666666666666)}


First, let's create a hypothetical QA dataset. In a real scenario, you would load this from a file or a library like Hugging Face `datasets`.

In [ ]:
# Hypothetical QA dataset (similar to SQuAD format simplified)
# Eventually, these will be replaced by actual fact checking question answers

qa_dataset = [
    {
        "id": "001",
        "question": "What is the capital of France?",
        "context": "Paris is the capital and most populous city of France.",
        "answers": [{"text": "Paris", "answer_start": 0}]
    },
    {
        "id": "002",
        "question": "Who invented the light bulb?",
        "context": "Thomas Edison is often credited with inventing the practical incandescent light bulb.",
        "answers": [{"text": "Thomas Edison", "answer_start": 0}]
    },
    {
        "id": "003",
        "question": "What is the largest planet in our solar system?",
        "context": "Jupiter is the largest planet in our solar system.",
        "answers": [{"text": "Jupiter", "answer_start": 0}]
    },
    {
        "id": "004",
        "question": "When was the internet invented?",
        "context": "The internet's origins date back to the development of packet switching in the 1960s.",
        "answers": [{"text": "1960s", "answer_start": 0}]
    }
]

print("Hypothetical QA Dataset:")
for item in qa_dataset:
    print(f"ID: {item['id']}, Question: {item['question']}, Answer: {item['answers'][0]['text']}")

Hypothetical QA Dataset:
ID: 001, Question: What is the capital of France?, Answer: Paris
ID: 002, Question: Who invented the light bulb?, Answer: Thomas Edison
ID: 003, Question: What is the largest planet in our solar system?, Answer: Jupiter
ID: 004, Question: When was the internet invented?, Answer: 1960s


Next, you would typically run your QA model to generate predictions for each question. For this example, we'll simulate some model predictions.

In [ ]:
# Simulate model predictions (these would come from your QA model)
# Eventually, these will be replaced by actual fact checking question answers

model_predictions = [
    {"id": "001", "prediction_text": "Paris"},
    {"id": "002", "prediction_text": "Edison"}, # Incorrect prediction
    {"id": "003", "prediction_text": "Jupiter"},
    {"id": "004", "prediction_text": "the 1960s"} # Partial match, might not be exact
]

print("\nSimulated Model Predictions:")
for item in model_predictions:
    print(f"ID: {item['id']}, Predicted Answer: {item['prediction_text']}")


Simulated Model Predictions:
ID: 001, Predicted Answer: Paris
ID: 002, Predicted Answer: Edison
ID: 003, Predicted Answer: Jupiter
ID: 004, Predicted Answer: the 1960s


In [ ]:
import evaluate

# Load the exact_match metric
exact_match_metric = evaluate.load("exact_match")

# Prepare lists for predictions and references
predictions_list = []
references_list = []

# Ensure predictions and references are aligned by ID
# A more robust approach would sort both lists by ID or use a dictionary lookup

# For simplicity, assuming both lists are already aligned by ID in this example
# (In a real scenario, you'd match by 'id' from qa_dataset and model_predictions)

# Manual alignment for demonstration based on the order of qa_dataset and model_predictions
# Ensure both lists are in the same order based on 'id'

qa_dataset_dict = {item['id']: item['answers'][0]['text'] for item in qa_dataset}
model_predictions_dict = {item['id']: item['prediction_text'] for item in model_predictions}

# Iterate through the common IDs to create aligned lists
common_ids = sorted(list(qa_dataset_dict.keys() & model_predictions_dict.keys()))

for qid in common_ids:
    references_list.append(qa_dataset_dict[qid])
    predictions_list.append(model_predictions_dict[qid])

print(f"\nReferences: {references_list}")
print(f"Predictions: {predictions_list}")

# Compute the exact match score
results = exact_match_metric.compute(predictions=predictions_list, references=references_list)

print("\nExact Match (EM) results for QA dataset:")
print(results)


References: ['Paris', 'Thomas Edison', 'Jupiter', '1960s']
Predictions: ['Paris', 'Edison', 'Jupiter', 'the 1960s']

Exact Match (EM) results for QA dataset:
{'exact_match': np.float64(0.5)}


ROUGE, BLEU, BERTSCORE (Semantic Similarity)

In [ ]:
from evaluate import load

# Load Metrics
rouge = load("rouge")
bleu = load("bleu")
bertscore = load("bertscore")

# Multiple predictions and references
predictions = [
    "The cat sat on the mat.",
    "The dog ran quickly through the park.",
    "I enjoy eating fresh apples."
]
references = [
    "A cat was sitting on the mat.",
    "A fast dog ran in the park.",
    "I like to eat fresh fruit."
]

# Compute ROUGE
rouge_scores = rouge.compute(predictions=predictions,references=references)

print(f"ROUGE-1: {rouge_scores['rouge1']}")

# Compute BLEU
bleu_scores = bleu.compute(predictions=predictions, references=[[ref] for ref in references]) # BLEU expects references as a list of lists
print("\nBLEU Score:")
print(bleu_scores)

# Compute BERTScore
bert_scores = bertscore.compute(predictions=predictions, references=references, lang="en")

print("\nBERTScore:")
print(bert_scores)

ROUGE-1: 0.5168165168165167

BLEU Score:
{'bleu': 0.22016663708986334, 'precisions': [0.6190476190476191, 0.3333333333333333, 0.2, 0.08333333333333333], 'brevity_penalty': 0.909156442876713, 'length_ratio': 0.9130434782608695, 'translation_length': 21, 'reference_length': 23}


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERTScore:
{'precision': [0.9757342338562012, 0.9518396854400635, 0.9656190276145935], 'recall': [0.9713872075080872, 0.9586963057518005, 0.9641715884208679], 'f1': [0.9735559225082397, 0.9552556276321411, 0.964894711971283], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.0.0)'}


### F1 Score for Question Answering using SQuAD v2 Metric

In [ ]:
import evaluate

# Load the squad_v2 metric
squad_metric = evaluate.load("squad_v2")

# Reuse the hypothetical QA dataset and model predictions
# qa_dataset and model_predictions are already defined in previous cells

# Format predictions for squad_v2 metric
# Each prediction needs 'id', 'prediction_text', and optionally 'no_answer_probability'
formatted_predictions = [
    {"id": item["id"], "prediction_text": item["prediction_text"], "no_answer_probability": 0.0}
    for item in model_predictions
]

# Format references for squad_v2 metric
# Each reference needs 'id' and an 'answers' dict with 'answer_start' and 'text' lists
formatted_references = [
    {
        "id": item["id"],
        "answers": {
            "answer_start": [item["answers"][0]["answer_start"]],
            "text": [item["answers"][0]["text"]]
        }
    }
    for item in qa_dataset
]

print("\nFormatted Predictions for SQuAD v2:")
for p in formatted_predictions:
    print(p)

print("\nFormatted References for SQuAD v2:")
for r in formatted_references:
    print(r)

# Compute the SQuAD v2 metric
results = squad_metric.compute(
    predictions=formatted_predictions,
    references=formatted_references
)

print("\nSQuAD v2 Metric Results (including F1 and Exact Match):")
print(results)


Formatted Predictions for SQuAD v2:
{'id': '001', 'prediction_text': 'Paris', 'no_answer_probability': 0.0}
{'id': '002', 'prediction_text': 'Edison', 'no_answer_probability': 0.0}
{'id': '003', 'prediction_text': 'Jupiter', 'no_answer_probability': 0.0}
{'id': '004', 'prediction_text': 'the 1960s', 'no_answer_probability': 0.0}

Formatted References for SQuAD v2:
{'id': '001', 'answers': {'answer_start': [0], 'text': ['Paris']}}
{'id': '002', 'answers': {'answer_start': [0], 'text': ['Thomas Edison']}}
{'id': '003', 'answers': {'answer_start': [0], 'text': ['Jupiter']}}
{'id': '004', 'answers': {'answer_start': [0], 'text': ['1960s']}}

SQuAD v2 Metric Results (including F1 and Exact Match):
{'exact': 75.0, 'f1': 91.66666666666666, 'total': 4, 'HasAns_exact': 75.0, 'HasAns_f1': 91.66666666666666, 'HasAns_total': 4, 'best_exact': 75.0, 'best_exact_thresh': 0.0, 'best_f1': 91.66666666666666, 'best_f1_thresh': 0.0}


Deep Eval

https://github.com/confident-ai/deepeval

In [ ]:
!pip install -U deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.7/840.7 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.4 MB/s eta 0:00:00


In [ ]:
!deepeval login

🔐 Enter your API Key: 
Saved environment variables to .env.local (ensure it's git-ignored).

🎉🥳 Congratulations! You've successfully logged in! 🙌
You're now using DeepEval with Confident AI. Follow our quickstart tutorial 
here: ]8;id=871082;https://www.confident-ai.com/docs/llm-evaluation/quickstart\https://www.confident-ai.com/docs/llm-evaluation/quickstart]8;;\


In [ ]:
!touch test_chatbot.py

In [ ]:
import pytest
from deepeval import assert_test
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

def test_case():
    correctness_metric = GEval(
        name="Correctness",
        criteria="Determine if the 'actual output' is correct based on the 'expected output'.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.5
    )
    test_case = LLMTestCase(
        input="What if these shoes don't fit?",
        # Replace this with the actual output from your LLM application
        actual_output="You have 30 days to get a full refund at no extra cost.",
        expected_output="We offer a 30-day full refund at no extra costs.",
        retrieval_context=["All customers are eligible for a 30 day full refund at no extra costs."]
    )
    assert_test(test_case, [correctness_metric])

In [ ]:
!deepeval test run test_chatbot.py

Running teardown with pytest sessionfinish...

3 warnings in 0.15s
No test cases found, please try again.


RAGAS Implementation

https://gist.github.com/donbr/a6e0fd539c1288b5ab3d88bb42f75a40

In [ ]:
!pip install ragas==0.2.15

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00


Basic RAGAS Setup

In [ ]:
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextRecall, FactualCorrectness, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

# Set up evaluator LLM
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o", temperature=0)
)

# Create rate-friendly configuration
rate_friendly_config = RunConfig(
    timeout=300,          # 5 minutes max for operations
    max_retries=15,       # More retries for rate limits
    max_wait=90,          # Longer wait between retries
    max_workers=8,        # Fewer concurrent API calls
    log_tenacity=True     # Log retry attempts
)

ragas_metrics=[
    LLMContextRecall(),
    Faithfulness(),
    FactualCorrectness(),
    ResponseRelevancy(),
    ContextEntityRecall(),
    NoiseSensitivity()
]

Choose Metrics

Implement Batch Processing

In [ ]:
def batch_process_evaluation(dataset, metrics, batch_size=5, pause_seconds=120):
    """Process evaluations in batches to avoid rate limits"""
    # Split dataset into batches
    batches = [dataset[i:i+batch_size] for i in range(0, len(dataset), batch_size)]
    print(f"Processing {len(dataset)} samples in {len(batches)} batches")

    all_results = []
    for i, batch in enumerate(batches):
        print(f"Processing batch {i+1}/{len(batches)}")

        # Run evaluation for this batch
        batch_result = evaluate(
            dataset=batch,
            metrics=ragas_metrics,
            llm=evaluator_llm,
            run_config=rate_friendly_config
        )

        all_results.append(batch_result)

        # Pause between batches (except after the last one)
        if i < len(batches) - 1:
            print(f"Pausing for {pause_seconds} seconds...")
            time.sleep(pause_seconds)

    return all_results

Use Multi-Tier Evaluation Approach

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
full_dataset = pd.read_csv('medreason-instruction-dataset.csv')

print(f"Loaded {len(full_dataset)} samples into full_dataset.")
# Display the first 5 rows of the loaded dataset
display(full_dataset.head())

Loaded 31535 samples into full_dataset.


,query,answer
0,Most sensitive test for H pylori,D. Urea breath test
1,Typhoid investigation of choice in 1st week,A. Blood culture
2,Urogenital Diaphragm is made up of the followi...,C. Colle's fascia
3,The technique for accurate quantification of g...,C. Real-Time Reverse Transcriptase PCR
4,Child with Type I Diabetes. What is the advise...,After 5 years


In [ ]:
from datasets import Dataset

# Assuming 'subset' DataFrame is already defined from previous steps
# If not, you might need to run the cell where 'subset' is created.

# Convert the pandas DataFrame 'subset' to a datasets.Dataset object
ragas_dataset_subset = Dataset.from_pandas(subset)

print("Converted 'subset' DataFrame to datasets.Dataset object:")
print(ragas_dataset_subset)

Converted 'subset' DataFrame to datasets.Dataset object:
Dataset({
    features: ['query', 'answer'],
    num_rows: 10
})


In [ ]:
from datasets import Dataset
import time
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextRecall, FactualCorrectness, ContextPrecision

# Helper function to retrieve contexts
def get_retrieved_contexts_for_ragas(query: str, k: int = 4) -> list[str]:
    """Retrieve relevant contexts from the vector database for a given query."""
    try:
        # Assuming vectordb is globally available and loaded
        docs = vectordb.max_marginal_relevance_search(query, k=k)
        if docs:
            return [doc.page_content for doc in docs]
        else:
            return []
    except Exception as e:
        print(f"Error retrieving contexts for query '{query}': {e}")
        return []

# Heavy metrics on small subset
heavy_metrics = [Faithfulness(), LLMContextRecall()]
subset_size = min(10, len(full_dataset))
subset_df = full_dataset.head(subset_size).copy() # Use .copy() to avoid SettingWithCopyWarning

# Prepare columns for RAGAS evaluation
subset_df['question'] = subset_df['query']
subset_df['contexts'] = subset_df['query'].apply(get_retrieved_contexts_for_ragas)
# For Faithfulness, 'response' should be the actual generated answer.
# For this initial setup, we'll use the 'answer' from the dataset as a placeholder for 'response'.
# In a full RAG pipeline, this 'response' would come from your LLM agent.
subset_df['response'] = subset_df['answer']
# RAGAS also often uses 'ground_truths' as a list of lists for multiple valid answers
# The 'reference' column expects a single string, not a list containing a string
subset_df['reference'] = subset_df['answer']

# Convert augmented subset DataFrame to datasets.Dataset for Ragas
ragas_dataset_subset = Dataset.from_pandas(subset_df)

heavy_results = evaluate(
    dataset=ragas_dataset_subset,
    metrics=heavy_metrics,
    llm=evaluator_llm,
    run_config=rate_friendly_config
)

# Prepare full_dataset for light metrics (and for light_results evaluation if needed)
full_dataset_copy = full_dataset.copy()
full_dataset_copy['question'] = full_dataset_copy['query']
full_dataset_copy['contexts'] = full_dataset_copy['query'].apply(get_retrieved_contexts_for_ragas)
full_dataset_copy['response'] = full_dataset_copy['answer'] # Placeholder for LLM generated response
# The 'reference' column expects a single string, not a list containing a string
full_dataset_copy['reference'] = full_dataset_copy['answer']

# Convert augmented full_dataset DataFrame to datasets.Dataset for Ragas
ragas_full_dataset = Dataset.from_pandas(full_dataset_copy)

# Lighter metrics on full dataset
light_metrics = [ResponseRelevancy(), ContextPrecision()] # ContextPrecision also needs 'question' and 'contexts'
light_results = batch_process_evaluation(
    dataset=ragas_full_dataset,
    metrics=light_metrics,
    batch_size=10,
    pause_seconds=60
)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no attribute 'replace'
Error retrieving contexts for query 'nan': 'float' object has no 

KeyboardInterrupt: 

TruLens Implementation

In [ ]:
!pip install trulens trulens_eval openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 87.1 MB/s eta 0:00:00


In [ ]:
!pip install trulens_eval langchain

In [ ]:
!pip install trulens-feedback

In [ ]:
!pip install trulens-providers-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import numpy as np
from trulens.core import Tru, Feedback, Select, TruSession
from trulens_eval.feedback.provider import OpenAI, Huggingface
from trulens_eval import Tru, Feedback, Select
from langchain_openai import ChatOpenAI , OpenAI as fOpenAI
from trulens_eval import Feedback

# Initialize TruLens
tru = Tru()

# Initialize OpenAI Feedback Class (using the same API key as your agent)
openai_feedback = fOpenAI()

# Define feedback functions
# You can customize these based on what aspects of your RAG you want to evaluate
# For example, relevance, groundedness, coherence, etc.

# Question/Answer Relevance (uses GPT-4o-mini to assess how relevant the answer is to the question)
#f_qa_relevance = Feedback(
#    openai_feedback.qa_relevance().on_input_output(),
#    name="Answer Relevance"
#).aggregate(np.mean)

# Initialize provider with your OpenAI API key
openai_provider = fOpenAI(api_key="OPENAI_API_KEY")

# Create a feedback function for context relevance
context_rel_feedback = Feedback(openai_provider.context_relevance())

# Context Relevance (assess how relevant retrieved context is to the question)
f_context_relevance = Feedback(
    openai_feedback.context_relevance().on_input_context().aggregate(np.mean),
    name="Context Relevance"
).aggregate(np.mean)

# Groundedness (assess how well the answer is supported by the context)
f_groundedness = Feedback(
    openai_feedback.groundedness().on_input_output().on(context=Select.RecordCalls.retrieve_context.rets).aggregate(np.mean),
    name="Groundedness"
).aggregate(np.mean)

# Optionally, add a sentiment feedback to check for positive/negative sentiment in the answer
f_sentiment = Feedback(
    openai_feedback.sentiment().on_output(),
    name="Sentiment"
).aggregate(np.mean)

# Combine feedback functions into a list
feedbacks = [f_qa_relevance, f_context_relevance, f_groundedness, f_sentiment]

AttributeError: 'OpenAI' object has no attribute 'context_relevance'